# LangChain RAG 系统构建学习Demo

本Notebook将深入学习LangChain中的RAG (Retrieval-Augmented Generation) 系统构建。

## 学习大纲
1. **RAG基础架构** - 文档加载、分割、向量化、检索
2. **RAG优化技术** - 多路召回、重排序、上下文压缩
3. **高级RAG模式** - Self-Query、Parent Document、Multi-Vector Retriever

## 环境准备

安装必要的依赖包

In [ ]:
# 安装依赖
# pip install langchain==0.3.15
# pip install langchain-core==0.3.28
# pip install langchain-community==0.3.14
# pip install dashscope==1.20.11
# pip install chromadb==0.5.23
# pip install faiss-cpu==1.9.0.post1
# pip install sentence-transformers==3.3.1
# pip install pypdf==5.1.0
# pip install lxml==5.3.0
# pip install rank-bm25==0.2.2

## 导入库并配置API Key

In [ ]:
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# 验证API Key
if DASHSCOPE_API_KEY:
    print("✅ API Key已加载")
else:
    print("❌ 请在.env文件中配置DASHSCOPE_API_KEY")

---

# 第一部分：RAG基础架构

RAG的核心流程包括：
1. **文档加载 (Document Loading)** - 从各种数据源加载文档
2. **文档分割 (Text Splitting)** - 将长文档切分成小块
3. **向量化 (Embedding)** - 将文本转换为向量
4. **向量存储 (Vector Store)** - 存储和索引向量
5. **检索 (Retrieval)** - 根据查询检索相关文档
6. **生成 (Generation)** - 基于检索结果生成答案

## 1.1 文档加载

LangChain支持多种文档加载器，包括文本文件、PDF、网页等。

In [ ]:
from langchain_core.documents import Document

# 创建示例文档
documents = [
    Document(
        page_content="LangChain是一个用于开发由大型语言模型驱动的应用程序的框架。它简化了LLM应用程序生命周期的每个阶段。",
        metadata={"source": "langchain_intro", "page": 1}
    ),
    Document(
        page_content="RAG (检索增强生成) 是一种结合了信息检索和文本生成的技术。它通过检索相关文档来增强LLM的响应质量。",
        metadata={"source": "rag_intro", "page": 1}
    ),
    Document(
        page_content="向量数据库用于存储和检索文档的向量表示。常见的向量数据库包括Chroma、FAISS、Pinecone等。",
        metadata={"source": "vector_db", "page": 1}
    ),
    Document(
        page_content="Embedding模型将文本转换为高维向量空间中的点。相似的文本在向量空间中距离较近。",
        metadata={"source": "embedding", "page": 1}
    ),
    Document(
        page_content="LangChain提供了多种文本分割器，如RecursiveCharacterTextSplitter、CharacterTextSplitter等，用于将长文档分割成小块。",
        metadata={"source": "text_splitter", "page": 1}
    )
]

print(f"📚 已加载 {len(documents)} 个文档")
print(f"\n示例文档:\n{documents[0].page_content}")

## 1.2 文档分割

文档分割是RAG中的关键步骤，需要平衡chunk大小和语义完整性。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建文本分割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # 每个chunk的最大字符数
    chunk_overlap=50,  # chunk之间的重叠字符数
    length_function=len,
    separators=["\n\n", "\n", "。", "！", "?", ",", " ", ""]
)

# 分割文档
splits = text_splitter.split_documents(documents)

print(f"📄 原始文档数: {len(documents)}")
print(f"✂️  分割后chunk数: {len(splits)}")
print(f"\n示例chunk:\n{splits[0].page_content}")
print(f"\nMetadata: {splits[0].metadata}")

## 1.3 向量化与存储

使用Embedding模型将文本转换为向量，并存储到向量数据库中。

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.vectorstores import Chroma

# 初始化Embedding模型（使用阿里云通义千问）
embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=DASHSCOPE_API_KEY
)

# 创建向量数据库
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="langchain_rag_demo"
)

print("✅ 向量数据库创建成功")
print(f"📊 存储了 {vectorstore._collection.count()} 个向量")

## 1.4 基础检索

使用向量相似度检索相关文档。

In [ ]:
# 创建检索器
retriever = vectorstore.as_retriever(
    search_type="similarity",  # 相似度搜索
    search_kwargs={"k": 3}  # 返回top-3结果
)

# 测试检索
query = "什么是RAG?"
retrieved_docs = retriever.invoke(query)

print(f"🔍 查询: {query}\n")
print(f"📖 检索到 {len(retrieved_docs)} 个相关文档:\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[{i}] {doc.page_content}")
    print(f"    来源: {doc.metadata}\n")

## 1.5 基础RAG链

将检索和生成组合成完整的RAG链。

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 初始化LLM
llm = ChatTongyi(
    model="qwen-plus",
    temperature=0.7,
    dashscope_api_key=DASHSCOPE_API_KEY
)

# 创建RAG提示词模板
rag_prompt = ChatPromptTemplate.from_template(
    """你是一个专业的技术助手。请基于以下上下文回答用户的问题。

上下文:
{context}

问题: {question}

请提供准确、详细的答案。如果上下文中没有相关信息，请说明这一点。
"""
)

# 格式化文档函数
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# 构建RAG链
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG链构建完成")

In [ ]:
# 测试RAG链
question = "什么是RAG？它有什么作用？"
answer = rag_chain.invoke(question)

print(f"❓ 问题: {question}\n")
print(f"💡 回答: {answer}")

---

# 第二部分：RAG优化技术

基础RAG虽然有效，但在复杂场景下可能存在以下问题：
- 检索精度不足
- 上下文噪声过多
- 无法处理复杂查询

我们将学习以下优化技术：
1. **混合检索 (Hybrid Retrieval)** - 结合向量检索和关键词检索
2. **重排序 (Reranking)** - 对检索结果进行二次排序
3. **上下文压缩 (Contextual Compression)** - 压缩和过滤检索结果

## 2.1 混合检索 (Hybrid Retrieval)

结合BM25关键词检索和向量语义检索，提高检索召回率。

In [ ]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# 创建BM25检索器（基于关键词）
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3

# 创建向量检索器
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 创建混合检索器
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]  # BM25权重0.4，向量检索权重0.6
)

# 测试混合检索
query = "文本分割器"
hybrid_results = ensemble_retriever.invoke(query)

print(f"🔍 混合检索查询: {query}\n")
print(f"📖 检索到 {len(hybrid_results)} 个文档:\n")
for i, doc in enumerate(hybrid_results, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

## 2.2 上下文压缩 (Contextual Compression)

使用压缩器从检索的文档中提取最相关的部分，减少噪声。

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# 创建LLM压缩器
compressor = LLMChainExtractor.from_llm(llm)

# 创建压缩检索器
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vector_retriever
)

# 测试压缩检索
query = "LangChain的主要用途是什么？"
compressed_docs = compression_retriever.invoke(query)

print(f"🔍 查询: {query}\n")
print("📦 压缩后的文档:\n")
for i, doc in enumerate(compressed_docs, 1):
    print(f"[{i}] {doc.page_content}")
    print()

## 2.3 MMR检索 (Maximal Marginal Relevance)

MMR平衡相关性和多样性，避免返回过于相似的文档。

In [ ]:
# 使用MMR检索
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,  # 先获取10个候选
        "lambda_mult": 0.7  # 多样性参数，0表示最大多样性，1表示最大相关性
    }
)

# 测试MMR检索
query = "向量数据库"
mmr_results = mmr_retriever.invoke(query)

print(f"🔍 MMR检索查询: {query}\n")
print(f"📖 检索结果 (平衡相关性和多样性):\n")
for i, doc in enumerate(mmr_results, 1):
    print(f"[{i}] {doc.page_content}")
    print()

---

# 第三部分：高级RAG模式

学习更高级的RAG架构模式：
1. **Self-Query Retriever** - 自动从查询中提取过滤条件
2. **Parent Document Retriever** - 检索小chunk，返回大chunk
3. **Multi-Vector Retriever** - 为同一文档创建多个向量表示

## 3.1 Parent Document Retriever

这种模式先用小chunk进行检索（提高精确度），然后返回包含该chunk的大文档（保持上下文完整性）。

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建父文档存储
docstore = InMemoryStore()

# 创建父文档分割器（大chunk）
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

# 创建子文档分割器（小chunk）
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

# 创建新的向量存储用于parent retriever
parent_vectorstore = Chroma(
    collection_name="parent_docs",
    embedding_function=embeddings
)

# 创建Parent Document Retriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 添加文档
parent_retriever.add_documents(documents)

print("✅ Parent Document Retriever创建成功")

# 测试检索
query = "LangChain框架"
parent_results = parent_retriever.invoke(query)

print(f"\n🔍 查询: {query}\n")
print(f"📖 检索结果 (返回完整父文档):\n")
for i, doc in enumerate(parent_results, 1):
    print(f"[{i}] {doc.page_content}")
    print(f"    长度: {len(doc.page_content)} 字符")
    print()

## 3.2 Multi-Vector Retriever

为同一文档生成多个向量表示（如摘要、问题等），提高检索的召回率。

In [ ]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
import uuid

# 创建文档ID
doc_ids = [str(uuid.uuid4()) for _ in documents]

# 创建多向量存储
multi_vectorstore = Chroma(
    collection_name="multi_vector_docs",
    embedding_function=embeddings
)

# 创建文档存储
multi_docstore = InMemoryStore()

# 存储原始文档
multi_docstore.mset(list(zip(doc_ids, documents)))

# 创建Multi-Vector Retriever
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=multi_vectorstore,
    docstore=multi_docstore,
    id_key="doc_id"
)

# 为每个文档生成摘要（这里简化为前50个字符）
summaries = []
for i, doc in enumerate(documents):
    summary_doc = Document(
        page_content=f"摘要: {doc.page_content[:50]}...",
        metadata={"doc_id": doc_ids[i]}
    )
    summaries.append(summary_doc)

# 添加摘要到向量库
multi_vectorstore.add_documents(summaries)

print("✅ Multi-Vector Retriever创建成功")
print(f"📊 原始文档数: {len(documents)}")
print(f"📊 向量表示数: {len(summaries)}")

# 测试检索
query = "向量"
multi_results = multi_vector_retriever.invoke(query)

print(f"\n🔍 查询: {query}\n")
print(f"📖 检索结果 (基于摘要检索，返回完整文档):\n")
for i, doc in enumerate(multi_results[:2], 1):
    print(f"[{i}] {doc.page_content}")
    print()

## 3.3 使用高级检索器构建优化的RAG链

In [ ]:
# 使用混合检索器构建优化的RAG链
advanced_rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 测试优化后的RAG链
question = "LangChain提供了哪些文本分割工具？"
answer = advanced_rag_chain.invoke(question)

print(f"❓ 问题: {question}\n")
print(f"💡 回答 (使用混合检索): {answer}")

---

# 总结

## 学到的内容

### 1. RAG基础架构
- ✅ 文档加载与处理
- ✅ 文本分割策略
- ✅ 向量化与存储
- ✅ 基础检索与生成

### 2. RAG优化技术
- ✅ 混合检索 (BM25 + Vector)
- ✅ 上下文压缩
- ✅ MMR检索 (平衡相关性和多样性)

### 3. 高级RAG模式
- ✅ Parent Document Retriever (小块检索，大块返回)
- ✅ Multi-Vector Retriever (多向量表示)

## 实践建议

1. **选择合适的chunk大小**：需要根据具体场景测试
2. **混合检索策略**：结合语义和关键词检索效果更好
3. **使用压缩器**：减少上下文噪声，提高生成质量
4. **Parent Document模式**：适合需要完整上下文的场景
5. **评估和迭代**：持续优化检索和生成效果

## 下一步学习

- 探索更多的Embedding模型
- 学习查询重写和扩展技术
- 研究RAG评估方法
- 尝试结合Agent进行多跳推理